# MTGFlow — le finestre più estreme del 2019

La soglia IQR dell'Eq. 13 è binaria: una finestra appena oltre e una dieci volte
oltre sono entrambe `is_anomaly`, e la pipeline di forecasting vede solo quel
flag. Qui si parte invece dallo **score continuo** per rispondere a tre domande:

1. quanto è lunga la coda, e dove cade la soglia rispetto ai quantili dell'anno?
2. quali giorni concentrano la coda estrema, senza sceglierli a priori dal calendario?
3. il forecaster degrada in modo monotono con lo score, o solo oltre una soglia?

Gli score non vengono mai mediati fra seed: si usa il singolo seed della run
densa che alimenta la pipeline.

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'physiq_pv').is_dir():
    raise FileNotFoundError('Avviare il notebook dalla root del repository o da notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import physiq_pv.reporting.anomaly_driver as anomaly_driver
import physiq_pv.reporting.anomaly_extremes as anomaly_extremes

anomaly_driver = importlib.reload(anomaly_driver)
anomaly_extremes = importlib.reload(anomaly_extremes)

RUN_NAME = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_seed1'
DETECTOR_SEED = 15
N_LOCATIONS = 1149

out_dir = ROOT / 'outputs' / RUN_NAME
scores_path = (ROOT / 'outputs' / 'pvgis_mtgflow' / 'downstream_dense'
               / f'seed_{DETECTOR_SEED}' / 'anomaly_scores.csv')
labels_path = out_dir / anomaly_driver.DRIVER_LABELS_FILE

for required in (scores_path, out_dir / 'predictions.csv'):
    if not required.is_file():
        raise FileNotFoundError(required)

labels = (
    anomaly_driver.load_driver_labels(labels_path) if labels_path.is_file() else None
)
print('Score detector  :', scores_path)
print('Etichette driver:',
      'caricate' if labels is not None else 'assenti (esegui prima il notebook driver)')

## 1. Forma della coda

Se la soglia IQR cade a un quantile basso, il flag binario è permissivo e lo
score continuo porta molta più informazione del flag.

In [ ]:
reference = anomaly_extremes.score_reference(scores_path)
display(pd.Series(reference).to_frame('valore').round(3))

threshold = reference.get('threshold_median')
scores = pd.read_csv(scores_path, usecols=['anomaly_score'])['anomaly_score']
if threshold is not None:
    print(f'Finestre oltre soglia: {np.mean(scores >= threshold):.3%}')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(scores, bins=200, histtype='step', linewidth=1.5)
if threshold is not None:
    ax.axvline(threshold, color='red', ls='--', label=f'soglia IQR = {threshold:.1f}')
for quantile in ('q0.99', 'q0.999'):
    ax.axvline(reference[quantile], color='black', ls=':', lw=1,
               label=f'{quantile} = {reference[quantile]:.1f}')
ax.set(yscale='log', xlabel='anomaly score', ylabel='finestre',
       title='Distribuzione degli score MTGFlow 2019')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 2. Giorni che concentrano la coda

I giorni sono ordinati per numero di finestre nella coda estrema dell'anno
(quantile `EXTREME_QUANTILE`), non per superamento della soglia binaria. La
composizione per driver arriva dalle etichette del notebook precedente.

In [ ]:
EXTREME_QUANTILE = 0.999

days = anomaly_extremes.rank_extreme_days(
    scores_path, quantile=EXTREME_QUANTILE, n_locations=N_LOCATIONS
)
display(anomaly_extremes.summarise_extreme_days(days, labels, top_n=20).round(3))

In [ ]:
top_windows = anomaly_extremes.top_extreme_windows(scores_path, k=200)
display(top_windows.head(20))

print('\nGiorni rappresentati nelle 200 finestre piu estreme:')
display(top_windows['timestamp'].dt.normalize().value_counts().head(10))
print('Localita distinte:', top_windows['location'].nunique())

## 3. Il forecaster degrada con lo score?

Il flag binario non distingue intensità. Qui le righe di previsione sono divise
in fasce di score crescente: se l'errore cresce in modo monotono, lo score
continuo è più informativo del flag e andrebbe usato come covariata invece che
come soglia.

In [ ]:
cut = reference.get('threshold_median', reference['q0.99'])
edges = [-np.inf, cut, reference['q0.99'], reference['q0.999'], np.inf]
names = ['sotto_soglia', 'soglia_q99', 'q99_q999', 'oltre_q999']

score_lookup = pd.read_csv(
    scores_path, usecols=['location', 'timestamp', 'anomaly_score'],
    dtype={'location': 'string'},
)
score_lookup['timestamp'] = pd.to_datetime(score_lookup['timestamp'])
score_lookup['location'] = score_lookup['location'].astype(str)
score_lookup['category'] = pd.cut(
    score_lookup['anomaly_score'], bins=edges, labels=names, right=False,
).astype(str)
score_lookup = score_lookup[['location', 'timestamp', 'category']]
display(score_lookup['category'].value_counts())

In [ ]:
band_comparison = anomaly_driver.build_anomaly_driver_comparison_figures(
    out_dir,
    score_lookup,
    figure_subdir='anomaly_score_bands',
    metrics_name='anomaly_score_band_metrics.csv',
    chunksize=500_000,
)
band_metrics = band_comparison['metrics']
display(band_metrics)

for metric in ('mae', 'rmse', 'bias', 'picp', 'nmpil'):
    print(f'\n=== {metric.upper()} ===')
    display(band_metrics.pivot(index='bin', columns='category', values=metric).round(3))

## 4. Lettura

- se `sotto_soglia` è già peggiore di `normal`, la soglia IQR lascia fuori
  condizioni che il forecaster sente comunque, e il flag binario perde informazione;
- se l'errore cresce da `soglia_q99` a `oltre_q999`, lo score è una **covariata di
  difficoltà** e non solo un rivelatore: è l'argomento per usarlo pesato invece che
  dicotomizzato;
- se il degrado è piatto oltre la soglia, dicotomizzare non perde nulla e il flag basta;
- i giorni in cima alla classifica vanno confrontati con quelli scelti a mano nei
  notebook evento: date diverse da aprile e giugno significano eventi non ancora
  analizzati.

Attenzione: le finestre coprono 60 ore, quindi un giorno in classifica può
ereditare l'estremo dal precedente. `window_start` e `window_end` restano nei CSV
per località se serve districare il trascinamento.